<a href="https://colab.research.google.com/github/harishkulkarni10/ai-research-assistant-rag/blob/dev/notebooks/04_retrieval/retrieval_faiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

# Vector retrieval with FAISS
- Let's build FAISS vector indices over chunk embeddings and retrieve the most relevant chunks for user queries using cosine similarity.

- Let's also compare retrieval behaviour between E5 and BGE embeddings using identical queries and index settings

In [ ]:
!pip install faiss-cpu --quiet

In [ ]:
from pathlib import Path
import os

bge_path = Path("/content/ai-research-assistant-rag/data/processed/chunks_bge.parquet")
e5_path  = Path("/content/ai-research-assistant-rag/data/processed/chunks_e5.parquet")

for path in [bge_path, e5_path]:
    if path.exists():
        print(f"\nFile: {path.name}")
        print(f"Size: {path.stat().st_size / 1e6:.2f} MB")

        # Check magic bytes (first 4 bytes of a real Parquet file)
        with open(path, 'rb') as f:
            magic = f.read(4)
            print(f"Magic bytes (first 4): {magic}")
            if magic == b'PAR1':
                print("→ Valid Parquet")
            else:
                print("→ NOT Parquet! Likely wrong format or corrupted")
    else:
        print(f"\nFile not found: {path}")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")

bge_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
e5_path  = PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet"

# Load both
chunks_bge = pd.read_parquet(bge_path)
chunks_e5  = pd.read_parquet(e5_path)

print(f"BGE chunks loaded: {len(chunks_bge):,} rows")
print(f"E5 chunks loaded: {len(chunks_e5):,} rows")
print(f"Columns in BGE: {list(chunks_bge.columns)}")
print(f"Embedding dim (BGE): {len(chunks_bge['embedding'].iloc[0])}")
print(f"Embedding dim (E5): {len(chunks_e5['embedding'].iloc[0])}")

In [ ]:
# Let's prepare vectors for FAISS

import numpy as np

bge_vectors = np.array(chunks_bge['embedding'].tolist(), dtype=np.float32)
e5_vectors  = np.array(chunks_e5['embedding'].tolist(), dtype=np.float32)

dim = bge_vectors.shape[1]
print(f"BGE dimension: {dim}")

dim2 = e5_vectors.shape[1]
print(f"E5 dimension: {dim2}")

In [ ]:
print(f"BGE vectors shape: {bge_vectors.shape}")
print(f"E5 vectors shape: {e5_vectors.shape}")
print(f"Data type: {bge_vectors.dtype}")

In [ ]:
# let's build FAISS indices - cosine similarity

import faiss

# BGE index
index_bge = faiss.IndexFlatIP(dim)
index_bge.add(bge_vectors)

index_e5 = faiss.IndexFlatIP(dim2)
index_e5.add(e5_vectors)

print("FAISS indices are built")

In [ ]:
# Let's load embedding model for queries

from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)
e5_model  = SentenceTransformer("intfloat/e5-base-v2", device=device)

print("Query embedding models loaded")

In [ ]:
# Define retrieval functions

def retrieve_bge(query: str, top_k: int = 5):
  q_emb = bge_model.encode(
      query,
      normalize_embeddings=True
  ).astype('float32')

  scores, indices = index_bge.search(
      q_emb.reshape(1, -2), top_k
  )
  return chunks_bge.iloc[indices[0]].assign(score = scores[0])

def retrieve_e5(query: str, top_k: int = 5):
    q_emb = e5_model.encode(
        "query: " + query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index_e5.search(
        q_emb.reshape(1, -2), top_k
    )
    return chunks_e5.iloc[indices[0]].assign(score = scores[0])


In [ ]:
query = "self-attention mechanism in transformer models"


print("=== BGE Retrieval ===")
bge_results = retrieve_bge(query, top_k=5)
display(bge_results[["chunk_id", "paper_id", "score", 'text']])

print("\n=== E5 Retrieval ===")
e5_results = retrieve_e5(query, top_k=5)
display(e5_results[["chunk_id", "paper_id", "score", 'text']])


In [ ]:
print("\nBGE Top Chunk Preview:\n")
print(bge_results.iloc[0]["text"][:1000])

print("\nE5 Top Chunk Preview:\n")
print(e5_results.iloc[0]["text"][:1000])


In [ ]:
# Define all 4 queries
queries = [
    "self-attention mechanism in transformer models",
    "quadratic complexity of self-attention",
    "positional encoding in transformers",
    "long sequence modeling transformer attention"
]

# Loop over each query
for i, query in enumerate(queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print(f"{'='*60}\n")

    # BGE Retrieval
    print("=== BGE Retrieval ===")
    bge_results = retrieve_bge(query, top_k=5)
    display(bge_results[["chunk_id", "paper_id", "score", "text"]])

    # E5 Retrieval
    print("\n=== E5 Retrieval ===")
    e5_results = retrieve_e5(query, top_k=5)
    display(e5_results[["chunk_id", "paper_id", "score", "text"]])

    # Top chunk previews
    print("\nBGE Top Chunk Preview:")
    print(bge_results.iloc[0]["text"][:1000] + "..." if len(bge_results.iloc[0]["text"]) > 1000 else bge_results.iloc[0]["text"])

    print("\nE5 Top Chunk Preview:")
    print(e5_results.iloc[0]["text"][:1000] + "..." if len(e5_results.iloc[0]["text"]) > 1000 else e5_results.iloc[0]["text"])

    print("\n")

# What exactly is our corpus really about
- Since we didn't get the required results (transformer-related, attention-related results, let's try to analyse what our corpus really has.
- We shortlisted a thousand papers randomly from the corpus of a million odd papers - so if we try to analyse the content of our shortlisted corpus, we may try asking right queries and know whether our retrieval is good or bad on the basis of that

In [ ]:
from collections import Counter
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Reload chunks (use BGE for this — doesn't matter)
PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
bge_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
chunks_df = pd.read_parquet(bge_path)

# Sample 50 random chunks for manual review
sample_chunks = chunks_df['text'].sample(50, random_state=42)
print("Sample chunk previews (first 200 chars each):")
for i, text in enumerate(sample_chunks, 1):
    print(f"{i}. {text[:200]}...\n")

# Common words analysis
vectorizer = CountVectorizer(stop_words='english', max_features=50)
word_counts = vectorizer.fit_transform(chunks_df['text'])
common_words = vectorizer.get_feature_names_out()
print("\nTop 50 most common words/phrases in corpus:")
print(common_words)

# Average text length
print(f"\nAverage chunk length: {chunks_df['text'].str.len().mean():.0f} chars")
print(f"Average token count: {chunks_df['token_count'].mean():.1f}")

In [ ]:
# Define all 5 queries - after we checked what our corpus really is about
queries = [
    "How are nonlinear dynamical systems modeled using differential equations?",
    "What methods are used to analyze stability and phase transitions in physical systems?",
    "How are time series and temporal dynamics analyzed in complex systems?",
    "What mathematical techniques are used for solving elliptic and partial differential equations?",
    "How are statistical distributions and energy models used in astrophysical systems?"
]

# Loop over each query
for i, query in enumerate(queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print(f"{'='*60}\n")

    # BGE Retrieval
    print("=== BGE Retrieval ===")
    bge_results = retrieve_bge(query, top_k=5)
    display(bge_results[["chunk_id", "paper_id", "score", "text"]])

    # E5 Retrieval
    print("\n=== E5 Retrieval ===")
    e5_results = retrieve_e5(query, top_k=5)
    display(e5_results[["chunk_id", "paper_id", "score", "text"]])

    # Top chunk previews
    print("\nBGE Top Chunk Preview:")
    print(bge_results.iloc[0]["text"][:1000] + "..." if len(bge_results.iloc[0]["text"]) > 1000 else bge_results.iloc[0]["text"])

    print("\nE5 Top Chunk Preview:")
    print(e5_results.iloc[0]["text"][:1000] + "..." if len(e5_results.iloc[0]["text"]) > 1000 else e5_results.iloc[0]["text"])

    print("\n")